In [5]:
import sys
from pathlib import Path
from scipy import stats
import plotly.express as px
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from dal.emergency_department_dal import EmergencyDepartmentDAL
from biz.emergency_department_analyzer import EmergencyDepartmentAnalyzer

In [6]:
objERDatabaseManager=EmergencyDepartmentDAL()
objEDAnalyzer=EmergencyDepartmentAnalyzer(objERDatabaseManager)

-> Connected to Central Database at: C:\Users\pavan\Python Projects\Pandas Projects\HeathCareAnalysis\data\ed_patient_data.db


In [7]:
dfPhysicianMetrics=objEDAnalyzer.fetchData()
dfPhysicianMetrics=dfPhysicianMetrics.loc[:,['attending_physician','wait_time_doctor_min','satisfaction_score','readmitted_30d','visit_id']]
#dfPhysicianMetrics
dfPhysicianMetricsGr=dfPhysicianMetrics.groupby("attending_physician",as_index=False).agg(
    avg_physician_wait_time=('wait_time_doctor_min','mean'),
    avg_physician_satisfaction_score=('satisfaction_score','mean'),
    patient_volume=('visit_id','count'),
    total_readmissions_30d=('readmitted_30d','sum')
)
dfPhysicianMetricsGr['rate_readmission_30d']=100.0 * dfPhysicianMetricsGr['total_readmissions_30d']/dfPhysicianMetricsGr['patient_volume']
dfPhysicianMetricsGr

,attending_physician,avg_physician_wait_time,avg_physician_satisfaction_score,patient_volume,total_readmissions_30d,rate_readmission_30d
0,Dr.AA,58.639526,6.228854,253,33,13.043478
1,Dr.AB,63.161004,6.223552,259,30,11.583012
2,Dr.BA,55.397119,6.254321,243,28,11.522634
3,Dr.BB,58.730418,6.176806,263,26,9.885932
4,Dr.CA,63.810121,6.021862,247,31,12.550607
5,Dr.CB,61.597244,6.088583,254,46,18.110236
6,Dr.DA,55.398374,6.223577,246,26,10.569106
7,Dr.DB,59.506178,6.215830,259,27,10.424710
8,Dr.EA,57.119008,6.121488,242,34,14.049587
9,Dr.EB,58.281496,6.267323,254,40,15.748031


**Do physicians with longer wait time have less satisfaction score?**
- Relation between physician wait time and satisfaction score
- Statastical Test Selection
    - Spearman Correlation ($\rho$) = "The Ranking Judge"

In [8]:
rho,p_value=stats.spearmanr(dfPhysicianMetricsGr["avg_physician_wait_time"],dfPhysicianMetricsGr["avg_physician_satisfaction_score"])
print(f"rho:{rho:.4f},p_value:{p_value:.04f}")


rho:-0.6559,p_value:0.0000


**Hypothesis Test: Physician Wait Time vs. Satisfaction Score**

**Null Hypothesis ($H_0$):**
- There is no relationship between physician wait time and patient satisfaction score.

**Alternative Hypothesis ($H_1$):**
- There is a statistically significant relationship between physician wait time and patient satisfaction score.

**Test Parameters & Results:**
- **Critical Threshold ($\alpha$):** `0.05`
- **P-Value:** `0.000` (or `< 0.001`)
- **Spearman Correlation ($\rho$):** `-0.6559`

**Conclusion ($p < \alpha$):**
- **Reject the Null Hypothesis:** Because the p-value (`0.000`) is less than the critical threshold (`0.05`), there is strong statistical evidence ofsignificant negative realtionship between physician wait times and satisfaction scores.
- Physician with longer average wait time tend to have lower satisfaction scores.
- **Moderate-to-Strong Negative Correlation ($\rho < 0$):** A negative correlation of `-0.6559` indicates that as physician wait time increases, patient satisfaction scores consistently decrease.


**Scatter Plot Visualiztion:Physician Wait Time vs. Satisfaction Score**

In [9]:
fig=px.scatter(
    data_frame=dfPhysicianMetricsGr,
    x="avg_physician_wait_time",
    y="avg_physician_satisfaction_score",
    title="Physician Wait Time vs. Satisfaction Score"
)
fig.show()

#### Do physicians attending high patient volume have less satisfaction score?
- Relation between patient volume and satisfaction score
- Statastical Test Selection
    - Spearman Correlation ($\rho$) = "The Ranking Judge"

In [10]:
rho,p_value=stats.spearmanr(dfPhysicianMetricsGr["patient_volume"],dfPhysicianMetricsGr["avg_physician_satisfaction_score"])
print(f"rho:{rho:.4f},p_value:{p_value:.04f}")

rho:0.0064,p_value:0.9688


**Hypothesis Test: Physician Patient Volume vs. Satisfaction Score**

**Null Hypothesis ($H_0$):**
- There is no relationship between a physician's patient volume and their average patient satisfaction score.

**Alternative Hypothesis ($H_1$):**
- There is a statistically significant relationship between a physician's patient volume and their average patient satisfaction score.

**Test Parameters & Results:**
- **Critical Threshold ($\alpha$):** `0.05`
- **P-Value:** `0.9688`
- **Spearman Correlation ($\rho$):** `0.0064`

**Conclusion ($p > \alpha$):**
- **Fail to Reject the Null Hypothesis:** Because the p-value (`0.9688`) is well above the critical threshold (`0.05`), there is not enough statistical evidence to conclude that patient volume impacts physician satisfaction scores.
- **No Meaningful Correlation ($\rho \approx 0$):** A Spearman coefficient of `0.0064` indicates virtually zero relationship—meaning busier physicians with higher patient volumes do not experience lower satisfaction ratings.

**Do physicians with high satisfaction score have lower 30d-readmissions?**
- Relation between physician satifaction score/rating and patient 30d-readmission
- Statastical Test Selection
    - Spearman Correlation ($\rho$) = "The Ranking Judge"

In [11]:
rho,p_value=stats.spearmanr(dfPhysicianMetricsGr["rate_readmission_30d"],dfPhysicianMetricsGr["avg_physician_satisfaction_score"])
print(f"rho:{rho:.4f},p_value:{p_value:.04f}")

rho:-0.0371,p_value:0.8200


**Hypothesis:** Do physician with higher satisfaction score have lower patient readmission 30 days

**Null Hypothesis($H\_0$):**

- There is no relationship behtween physician satisfaction score and patient readmission 30d
- Physicians with high patient satisfaction score have no effect or impact on patient readmissions 30 days


**Alternate Hypothesis($H\_1$):**

- There is relationship betweeen physician satisfaction score and patient readmission 30d
- Physicians with high patient satisfaction score have impact on patient readmissions 30 days
**Test Parameters & Results:**

- Critical Threshold($\alpha$):0.05
- pvalue:0.820
- Spearman Correlation($\rho$):-0.0371


**($p-value > \alpha$):**

- Failed to reject Null Hypothesis
- Because p-value (0.82) is well above the critical threshold there is not enough statastical evidence to conclude that physician satification score have different patient readmission 30 days.
- There is not enough statastical evidence that physicians with higher satisfaction scores tend to have lower patient readmissions 30 days

**No Meaningful Correlation ($\rho \approx  0$):**

- A spearman coefficient -0.0371 indicates virtually zero relationship
- Physicians with higher satisfaction score donot have lower patient readmissions 30 days.

**Business Insight:**

- Although physician satisfaction score is an important experience metric this analysis found no statastical significance between physician satisfaction score and patient 30 day readmission.The data suggests that factors influencing readmissions may differ from those driving patient satisfaction.

**Do physicians with longer wait times also experience higher 30-day readmission rates?**
- Relation between physician wait times and 30-day readmission rates
- Statastical Test Selection
    - Spearman Correlation ($\rho$) = "The Ranking Judge"

In [12]:
rho,p_value=stats.spearmanr(dfPhysicianMetricsGr["rate_readmission_30d"],dfPhysicianMetricsGr["avg_physician_wait_time"])
print(f"rho:{rho:.4f},p_value:{p_value:.04f}")

rho:-0.0660,p_value:0.6856


**Hypothesis:** Do physician with longer wait times experience higher readmission 30 days

**Null Hypothesis($H\_0$):**

- There is no relationship behtween physician wait time and patient readmission 30d
- Physicians with longer wait times have no effect on patient readmissions 30 days


**Alternate Hypothesis($H\_1$):**

- There is relationship betweeen physician wait time  and patient readmission 30d
- Physicians with longer wait times have  effect on patient readmissions 30 days

**Test Parameters & Results:**

- Critical Threshold($\alpha$):0.05
- pvalue:0.6856
- Spearman Correlation($\rho$):-0.0660


**($p-value > \alpha$) :**

- Failed to reject Null Hypothesis
- Because p-value (0.6856) is well above the critical threshold there is not enough statastical evidence to conclude that physician wait time influences the  patient readmission 30 days.
- There is not enough statastical evidence that physicians with longer wait times tend to have higer patient readmissions 30 days

**No Meaningful Correlation ($\rho \approx  0$):**

- A spearman coefficient -0.0660 indicates virtually zero relationship
- Physicians with longer wait times donot experience higher patient readmissions 30 days


In [13]:
dfPhysicianMetricsOverview=objEDAnalyzer.get_physician_performance_metrics()
dfPhysicianMetricsOverview

,attending_physician,physician_wait_time,physician_critical_wait_time,physician_satisfaction_score,physician_critical_satisfaction_score,readmission_rate_30d,critical_readmission_rate_30d,patient_volume,critical_patient_volume
0,Dr.AA,67.00,21.02,6.05,7.05,8.70,32.61,207,46
1,Dr.AB,69.75,30.98,6.10,6.83,7.44,31.82,215,44
2,Dr.BA,60.42,26.53,6.12,7.02,10.14,19.44,207,36
3,Dr.BB,65.51,26.73,6.02,6.94,8.29,17.39,217,46
4,Dr.CA,70.15,30.01,5.88,6.78,11.54,17.95,208,39
5,Dr.CB,69.87,25.17,5.91,6.89,14.49,34.04,207,47
6,Dr.DA,63.26,22.10,6.09,6.79,8.04,21.28,199,47
7,Dr.DB,66.23,24.76,6.05,7.05,7.37,26.19,217,42
8,Dr.EA,64.96,21.84,5.95,6.89,10.10,31.82,198,44
9,Dr.EB,67.01,25.97,6.07,6.99,12.50,27.78,200,54


In [14]:
dfPhysicianMetricsM=dfPhysicianMetricsOverview.loc[:,['attending_physician','critical_patient_volume']]
dfPhysicianMetricsMerged=pd.merge(dfPhysicianMetricsM, dfPhysicianMetricsGr, on='attending_physician', how='inner')
dfPhysicianMetricsMerged

,attending_physician,critical_patient_volume,avg_physician_wait_time,avg_physician_satisfaction_score,patient_volume,total_readmissions_30d,rate_readmission_30d
0,Dr.AA,46,58.639526,6.228854,253,33,13.043478
1,Dr.AB,44,63.161004,6.223552,259,30,11.583012
2,Dr.BA,36,55.397119,6.254321,243,28,11.522634
3,Dr.BB,46,58.730418,6.176806,263,26,9.885932
4,Dr.CA,39,63.810121,6.021862,247,31,12.550607
5,Dr.CB,47,61.597244,6.088583,254,46,18.110236
6,Dr.DA,47,55.398374,6.223577,246,26,10.569106
7,Dr.DB,42,59.506178,6.215830,259,27,10.424710
8,Dr.EA,44,57.119008,6.121488,242,34,14.049587
9,Dr.EB,54,58.281496,6.267323,254,40,15.748031


**Do physicians with higher critical patient volume experience higher wait times?**
- Relation between physician critical patient volume and 
- Statastical Test Selection
    - Spearman Correlation ($\rho$) = "The Ranking Judge"

In [15]:
rho,p_value=stats.spearmanr(dfPhysicianMetricsMerged['critical_patient_volume'],dfPhysicianMetricsMerged['avg_physician_wait_time'])
print(f"rho:{rho:.4f},p-value:{p_value:.4f}")

rho:-0.1556,p-value:0.3377


**Null Hypothesis($H\_0$):**

- There is no relationship between physician critical patient volume and physician wait time
- Physicians with higher critical patient volume have no effect on physician wait time


**Alternate Hypothesis($H\_1$):**

- There is relationship betweeen physician critical patient volume and physician wait time

* Physicians with higher critical patient volume have an effect on physician wait time

**Results:**

- Critical Threshold($\alpha$):0.05
- pvalue:0.3377
- Spearman Correlation($\rho$):-0.1556


**($p-value > \alpha$) :**

- Failed to reject Null Hypothesis
- Because p-value (0.3377) is well above the critical threshold there is not enough statastical evidence to conclude that there is an association between physician critical patient volume and physicians wait time.
- There is not enough statastical evidence that physicians with higher critical patient volume tend to have longer wait time

**No Meaningful Correlation ($\rho \approx  0$):**

- A spearman coefficient -0.1556 indicates virtually zero relationship.
- There is not enough statastical evidence that physicians attending larger critcal patients have longer wait times.

**Business Insight:**

- Although physician wait time is an important physician efficiancy metric .This analysis found no statastical significance between number of critical pattient physician is attending and physician wait time.The data suggests there are factors other than physician critcal patient volume that drives the physician wait time


<br />



| Question | Result | Business Outcome |
|:---|:---|:---|
| Physician Wait-Time vs patient satisfaction | Strong Negative | Significant Driver |
| Physician patient volume vs patient satisfaction | No relationship | Not Driver​​​​ |
| physician satisfaction vs readmission rate | No relationship | Not Driver |
| physician wait time vs readmission rate | No relationship | Nor Driver |
| physician critical patient volumne vs wait time | No relationship | Nor Driver |


**Business Insights:**

Physician wait time appears to be an important driver of patient experience but not a predictor of 30-day readmission outcomes. While longer wait times were associated with lower patient satisfaction, this analysis found no evidence that they are associated with increased readmission rates. This suggests that factors affecting patient experience may differ from those influencing longer-term clinical outcomes.

**Business Question**

Do critical patients report lower satisfaction scores than non-critical patients?

In [16]:
t_stats,p_value=stats.ttest_ind(
        dfPhysicianMetricsOverview['physician_satisfaction_score'],
        dfPhysicianMetricsOverview['physician_critical_satisfaction_score'],
        equal_var=False,
        nan_policy='omit'
)
print(f"t_stats:{t_stats:.4f},p_value:{p_value:.4f}")
print(f"non critical satisfaction:{dfPhysicianMetricsOverview['physician_satisfaction_score'].mean()}")
print(f"critical satisfaction:{dfPhysicianMetricsOverview['physician_critical_satisfaction_score'].mean()}")

t_stats:-38.7418,p_value:0.0000
non critical satisfaction:6.005750000000001
critical satisfaction:6.9197500000000005


**Hypothesis:** Do critical patients report lower satisfaction scores than non-critical patients?

**Null Hypothesis($H\_0$):**

- There is change is critical and non critical patient satisfaction

- Mathematically:
  - Mean (Critical Patient Satisfaction Score) = Mean (Non Critical Patient Satisfaction Score)


**Alternate Hypothesis($H\_1$):**

- There is difference in satisfaction score reported by critical patients and non critical patients

* Mathematically:
  - Mean(Critical Patient Satisfaction Score) `<>` Mean (Non Critical Patient Satisfaction Score)

**Results:**

- Critical Threshold($\alpha$):0.05
- pvalue:0.0000
- $t\_stats:-38.7418$


**($p-value < \alpha$) :**

- Reject Null Hypothesis
- Because p-value (0.0000) is well belown the critical threshold there is enough statastical evidence to conclude that there is difference in satisfaction scores reported by critical and non critical patients.

**Negative Two Sample Test TTest ($t\_stats$**):

- Critical patients reported significantly higher satisfaction scores (Mean = 6.92) than non-critical patients (Mean = 6.01).

**Business Insight:**

- Critical patients reported significantly higher satisfaction scores than non-critical patients. This suggests that emergency department processes, prioritization protocols, or the perceived value of care received during high-acuity situations may contribute to a more positive patient experience among critical cases.

Business Question

Are some physicians consistently better across multiple dimensions of performance?

In [17]:
#dfPhysicianMetricsMerged.columns
#Large patient volume
dfPhysicianRanked=dfPhysicianMetricsMerged.copy()
dfPhysicianRanked['patient_volume_rnk']=dfPhysicianRanked['patient_volume'].rank(
    method="dense",
    ascending=False
)
#Highest patient satisfaction
dfPhysicianRanked['patient_satisfaction_rnk']=dfPhysicianRanked['avg_physician_satisfaction_score'].rank(
    method="dense",
    ascending=False
)
#Lowest readmission rate
dfPhysicianRanked['patient_readmission_rnk']=dfPhysicianRanked['rate_readmission_30d'].rank(
    method="dense",
    ascending=True
)
#Lowest wait time
dfPhysicianRanked['physician_wait_time_rnk']=dfPhysicianRanked['avg_physician_wait_time'].rank(
    method="dense",
    ascending=True
)
dfPhysicianRanked=dfPhysicianRanked.loc[:,['attending_physician','patient_volume_rnk','patient_satisfaction_rnk','patient_readmission_rnk','physician_wait_time_rnk']]
dfPhysicianRanked['combinedRank']=(
    dfPhysicianRanked['patient_satisfaction_rnk']
    + dfPhysicianRanked['patient_readmission_rnk']
    + dfPhysicianRanked['physician_wait_time_rnk']
)
dfPhysicianRanked=dfPhysicianRanked.loc[:,['attending_physician','combinedRank','patient_volume_rnk','patient_satisfaction_rnk','patient_readmission_rnk','physician_wait_time_rnk']]
dfPhysicianRanked=dfPhysicianRanked.sort_values(by="combinedRank",ascending=True)
dfPhysicianRanked

,attending_physician,combinedRank,patient_volume_rnk,patient_satisfaction_rnk,patient_readmission_rnk,physician_wait_time_rnk
25,Dr.MB,9.0,13.0,2.0,1.0,6.0
2,Dr.BA,23.0,20.0,7.0,13.0,3.0
19,Dr.JB,24.0,17.0,3.0,16.0,5.0
6,Dr.DA,24.0,17.0,10.0,10.0,4.0
36,Dr.WA,27.0,14.0,1.0,25.0,1.0
23,Dr.LB,31.0,3.0,8.0,6.0,17.0
32,Dr.SA,32.0,16.0,5.0,20.0,7.0
7,Dr.DB,37.0,9.0,12.0,7.0,18.0
3,Dr.BB,40.0,5.0,23.0,4.0,13.0
30,Dr.QA,42.0,4.0,22.0,18.0,2.0


In [ ]:
fig = px.scatter(
    dfPhysicianMetricsMerged,
    x="avg_physician_wait_time",
    y="avg_physician_satisfaction_score",
    size="patient_volume",
    color="rate_readmission_30d",
    color_continuous_scale=["green", "red"],
    size_max=8,
    hover_name="attending_physician",
    hover_data={
        "avg_physician_wait_time": ":.2f",
        "avg_physician_satisfaction_score": ":.2f",
        "patient_volume": True,
        "rate_readmission_30d": ":.2f",
    },
    labels={
        "avg_physician_wait_time": "Average wait time (minutes)",
        "avg_physician_satisfaction_score": "Average satisfaction score",
        "patient_volume": "Patient volume",
        "rate_readmission_30d": "30-day readmission rate (%)",
    },
    title="Physician Performance Benchmarking: Wait Time vs Patient Satisfaction",
)

# Use medians to divide physicians into quadrants.
fig.add_vline(
    x=dfPhysicianMetricsMerged["avg_physician_wait_time"].median(),
    line_dash="dash",
    line_color="gray",
)
fig.add_hline(
    y=dfPhysicianMetricsMerged["avg_physician_satisfaction_score"].median(),
    line_dash="dash",
    line_color="gray",
)

fig.show()